In [0]:
patients_df = spark.table("bronze_patients")
doctors_df = spark.table("bronze_doctors")
appointments_df = spark.table("bronze_appointments")
preferences_df = spark.table("bronze_preferences")

In [0]:
preferences_flat = preferences_df.select(
    "patient_id",
    "preferred_hospital",
    preferences_df.contact.phone.alias("phone"),
    preferences_df.contact.email.alias("email")
)

display(preferences_flat)

patient_id,preferred_hospital,phone,email
P101,Apollo Hospital,9876500011,rahul@mail.com
P102,Yashoda Hospital,9876500012,priya@mail.com
P104,Care Hospital,9876500014,sneha@mail.com
P108,Apollo Hospital,9876500018,meera@mail.com


In [0]:
from pyspark.sql.functions import *

patients_df = patients_df.withColumn(
    "age",
    col("age").cast("int")
)

doctors_df = doctors_df.withColumn(
    "consultation_fee",
    col("consultation_fee").cast("double")
)

appointments_df = appointments_df.withColumn(
    "bill_amount",
    col("bill_amount").cast("double")
)

In [0]:
patient_pref_df = patients_df.join(
    preferences_flat,
    "patient_id",
    "left"
)

display(patient_pref_df)

patient_id,patient_name,city,state,age,gender,insurance_status,preferred_hospital,phone,email
P101,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com
P102,Priya Reddy,Bangalore,Karnataka,29,Female,Active,Yashoda Hospital,9876500012,priya@mail.com
P103,Amit Kumar,Mumbai,Maharashtra,42,Male,Inactive,null,null,null
P104,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com
P105,Farhan Ali,Chennai,Tamil Nadu,55,Male,Active,null,null,null
P106,Neha Singh,Pune,Maharashtra,38,Female,Inactive,null,null,null
P107,Arjun Verma,Hyderabad,Telangana,26,Male,Active,null,null,null
P108,Meera Nair,Kochi,Kerala,48,Female,Active,Apollo Hospital,9876500018,meera@mail.com


In [0]:
appt_patient_df = appointments_df.join(
    patient_pref_df,
    "patient_id"
)

display(appt_patient_df)

patient_id,appointment_id,doctor_id,appointment_date,diagnosis,bill_amount,status,patient_name,city,state,age,gender,insurance_status,preferred_hospital,phone,email
P101,A1009,D106,2026-06-05,Cardiac Review,6500.0,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com
P102,A1002,D102,2026-06-01,Migraine,3500.0,Completed,Priya Reddy,Bangalore,Karnataka,29,Female,Active,Yashoda Hospital,9876500012,priya@mail.com
P103,A1003,D103,2026-06-02,Skin Allergy,2000.0,Pending,Amit Kumar,Mumbai,Maharashtra,42,Male,Inactive,null,null,null
P104,A1010,D104,2026-06-05,Back Pain,4500.0,Cancelled,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com
P105,A1005,D105,2026-06-03,Fever,1500.0,Completed,Farhan Ali,Chennai,Tamil Nadu,55,Male,Active,null,null,null
P106,A1006,D106,2026-06-03,Heart Checkup,7000.0,Completed,Neha Singh,Pune,Maharashtra,38,Female,Inactive,null,null,null
P107,A1007,D101,2026-06-04,Chest Pain,5500.0,Completed,Arjun Verma,Hyderabad,Telangana,26,Male,Active,null,null,null
P108,A1008,D103,2026-06-04,Skin Infection,2500.0,Pending,Meera Nair,Kochi,Kerala,48,Female,Active,Apollo Hospital,9876500018,meera@mail.com
P101,A1001,D101,2026-06-01,Heart Checkup,5000.0,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com
P104,A1004,D104,2026-06-02,Fracture,12000.0,Completed,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com


In [0]:
final_df = appt_patient_df.join(
    doctors_df,
    "doctor_id"
)

display(final_df)

doctor_id,patient_id,appointment_id,appointment_date,diagnosis,bill_amount,status,patient_name,city,state,age,gender,insurance_status,preferred_hospital,phone,email,doctor_name,department,city,consultation_fee
D101,P101,A1001,2026-06-01,Heart Checkup,5000.0,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Ramesh,Cardiology,Hyderabad,1500.0
D102,P102,A1002,2026-06-01,Migraine,3500.0,Completed,Priya Reddy,Bangalore,Karnataka,29,Female,Active,Yashoda Hospital,9876500012,priya@mail.com,Dr. Priya,Neurology,Bangalore,2000.0
D103,P103,A1003,2026-06-02,Skin Allergy,2000.0,Pending,Amit Kumar,Mumbai,Maharashtra,42,Male,Inactive,null,null,null,Dr. Anita,Dermatology,Chennai,1000.0
D104,P104,A1004,2026-06-02,Fracture,12000.0,Completed,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500.0
D105,P105,A1005,2026-06-03,Fever,1500.0,Completed,Farhan Ali,Chennai,Tamil Nadu,55,Male,Active,null,null,null,Dr. Meera,Pediatrics,Delhi,1200.0
D106,P106,A1006,2026-06-03,Heart Checkup,7000.0,Completed,Neha Singh,Pune,Maharashtra,38,Female,Inactive,null,null,null,Dr. Kiran,Cardiology,Hyderabad,3000.0
D101,P107,A1007,2026-06-04,Chest Pain,5500.0,Completed,Arjun Verma,Hyderabad,Telangana,26,Male,Active,null,null,null,Dr. Ramesh,Cardiology,Hyderabad,1500.0
D103,P108,A1008,2026-06-04,Skin Infection,2500.0,Pending,Meera Nair,Kochi,Kerala,48,Female,Active,Apollo Hospital,9876500018,meera@mail.com,Dr. Anita,Dermatology,Chennai,1000.0
D106,P101,A1009,2026-06-05,Cardiac Review,6500.0,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Kiran,Cardiology,Hyderabad,3000.0
D104,P104,A1010,2026-06-05,Back Pain,4500.0,Cancelled,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500.0


In [0]:
final_df = final_df.withColumn(
    "final_bill",
    col("bill_amount") + col("consultation_fee")
)

In [0]:
final_df = final_df.withColumn(
    "appointment_month",
    month("appointment_date")
)

In [0]:
from pyspark.sql.functions import when

final_df = final_df.withColumn(
    "patient_age_group",
    when(col("age") >= 50, "Senior")
    .when(col("age") >= 30, "Adult")
    .otherwise("Young")
)

In [0]:
display(final_df)

doctor_id,patient_id,appointment_id,appointment_date,diagnosis,bill_amount,status,patient_name,city,state,age,gender,insurance_status,preferred_hospital,phone,email,doctor_name,department,city,consultation_fee,final_bill,appointment_month,patient_age_group
D101,P101,A1001,2026-06-01,Heart Checkup,5000.0,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Ramesh,Cardiology,Hyderabad,1500.0,6500.0,6,Adult
D102,P102,A1002,2026-06-01,Migraine,3500.0,Completed,Priya Reddy,Bangalore,Karnataka,29,Female,Active,Yashoda Hospital,9876500012,priya@mail.com,Dr. Priya,Neurology,Bangalore,2000.0,5500.0,6,Young
D103,P103,A1003,2026-06-02,Skin Allergy,2000.0,Pending,Amit Kumar,Mumbai,Maharashtra,42,Male,Inactive,null,null,null,Dr. Anita,Dermatology,Chennai,1000.0,3000.0,6,Adult
D104,P104,A1004,2026-06-02,Fracture,12000.0,Completed,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500.0,14500.0,6,Adult
D105,P105,A1005,2026-06-03,Fever,1500.0,Completed,Farhan Ali,Chennai,Tamil Nadu,55,Male,Active,null,null,null,Dr. Meera,Pediatrics,Delhi,1200.0,2700.0,6,Senior
D106,P106,A1006,2026-06-03,Heart Checkup,7000.0,Completed,Neha Singh,Pune,Maharashtra,38,Female,Inactive,null,null,null,Dr. Kiran,Cardiology,Hyderabad,3000.0,10000.0,6,Adult
D101,P107,A1007,2026-06-04,Chest Pain,5500.0,Completed,Arjun Verma,Hyderabad,Telangana,26,Male,Active,null,null,null,Dr. Ramesh,Cardiology,Hyderabad,1500.0,7000.0,6,Young
D103,P108,A1008,2026-06-04,Skin Infection,2500.0,Pending,Meera Nair,Kochi,Kerala,48,Female,Active,Apollo Hospital,9876500018,meera@mail.com,Dr. Anita,Dermatology,Chennai,1000.0,3500.0,6,Adult
D106,P101,A1009,2026-06-05,Cardiac Review,6500.0,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Kiran,Cardiology,Hyderabad,3000.0,9500.0,6,Adult
D104,P104,A1010,2026-06-05,Back Pain,4500.0,Cancelled,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500.0,7000.0,6,Adult


In [0]:
from pyspark.sql.functions import col

doctors_df_renamed = doctors_df.withColumnRenamed("city", "doctor_city")

final_df = appointments_df \
    .join(patient_pref_df, "patient_id") \
    .join(doctors_df_renamed, "doctor_id")

In [0]:
final_df = final_df.withColumn(
    "final_bill",
    col("bill_amount") + col("consultation_fee")
)

final_df = final_df.withColumn(
    "appointment_month",
    month("appointment_date")
)

final_df = final_df.withColumn(
    "patient_age_group",
    when(col("age") >= 50, "Senior")
    .when(col("age") >= 30, "Adult")
    .otherwise("Young")
)

In [0]:
display(final_df)

doctor_id,patient_id,appointment_id,appointment_date,diagnosis,bill_amount,status,patient_name,city,state,age,gender,insurance_status,preferred_hospital,phone,email,doctor_name,department,doctor_city,consultation_fee,final_bill,appointment_month,patient_age_group
D101,P101,A1001,2026-06-01,Heart Checkup,5000.0,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Ramesh,Cardiology,Hyderabad,1500.0,6500.0,6,Adult
D102,P102,A1002,2026-06-01,Migraine,3500.0,Completed,Priya Reddy,Bangalore,Karnataka,29,Female,Active,Yashoda Hospital,9876500012,priya@mail.com,Dr. Priya,Neurology,Bangalore,2000.0,5500.0,6,Young
D103,P103,A1003,2026-06-02,Skin Allergy,2000.0,Pending,Amit Kumar,Mumbai,Maharashtra,42,Male,Inactive,null,null,null,Dr. Anita,Dermatology,Chennai,1000.0,3000.0,6,Adult
D104,P104,A1004,2026-06-02,Fracture,12000.0,Completed,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500.0,14500.0,6,Adult
D105,P105,A1005,2026-06-03,Fever,1500.0,Completed,Farhan Ali,Chennai,Tamil Nadu,55,Male,Active,null,null,null,Dr. Meera,Pediatrics,Delhi,1200.0,2700.0,6,Senior
D106,P106,A1006,2026-06-03,Heart Checkup,7000.0,Completed,Neha Singh,Pune,Maharashtra,38,Female,Inactive,null,null,null,Dr. Kiran,Cardiology,Hyderabad,3000.0,10000.0,6,Adult
D101,P107,A1007,2026-06-04,Chest Pain,5500.0,Completed,Arjun Verma,Hyderabad,Telangana,26,Male,Active,null,null,null,Dr. Ramesh,Cardiology,Hyderabad,1500.0,7000.0,6,Young
D103,P108,A1008,2026-06-04,Skin Infection,2500.0,Pending,Meera Nair,Kochi,Kerala,48,Female,Active,Apollo Hospital,9876500018,meera@mail.com,Dr. Anita,Dermatology,Chennai,1000.0,3500.0,6,Adult
D106,P101,A1009,2026-06-05,Cardiac Review,6500.0,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Kiran,Cardiology,Hyderabad,3000.0,9500.0,6,Adult
D104,P104,A1010,2026-06-05,Back Pain,4500.0,Cancelled,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500.0,7000.0,6,Adult


In [0]:
final_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("silver_healthcare")

In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,bronze_appointments,false
default,bronze_doctors,false
default,bronze_patients,false
default,bronze_preferences,false
default,gold_retail,false
default,managed_retail,false
default,retail_sql,false
default,silver_healthcare,false
